# Notebook 2: Data Cleaning & Master Table
Build one clean row per delivered order, with delivery, payment and review details.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(parents=True, exist_ok=True)   # creates data/processed if it doesn't exist

orders    = pd.read_csv(RAW / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
items     = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments  = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews   = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
print("Tables loaded")

Tables loaded


In [2]:
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders[date_cols].dtypes

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [3]:
print(orders.isna().sum())
print("\nStatus of orders with no delivery date:")
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Status of orders with no delivery date:


order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [5]:
delivered = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
].copy()

print("All orders:      ", len(orders))
print("Delivered orders:", len(delivered))

All orders:       99441
Delivered orders: 96470


In [6]:
delivered_date = delivered["order_delivered_customer_date"].dt.normalize()   # drop the time, keep the date
estimated_date = delivered["order_estimated_delivery_date"].dt.normalize()

delivered["delivery_days"] = (delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]).dt.days
delivered["delay_days"]    = (delivered_date - estimated_date).dt.days        # positive = late, negative = early
delivered["is_late"]       = delivered_date > estimated_date

print("Late delivery rate:", round(delivered["is_late"].mean() * 100, 2), "%")
delivered[["delivery_days", "delay_days"]].describe()

Late delivery rate: 6.77 %


,delivery_days,delay_days
count,96470.000000,96470.000000
mean,12.093604,-11.875889
std,9.551380,10.182105
min,0.000000,-147.000000
25%,6.000000,-17.000000
50%,10.000000,-12.000000
75%,15.000000,-7.000000
max,209.000000,188.000000


In [7]:
items_per_order = items.groupby("order_id").agg(
    n_items       = ("order_item_id", "count"),
    product_value = ("price", "sum"),
    freight_value = ("freight_value", "sum"),
    n_sellers     = ("seller_id", "nunique"),
).reset_index()

items_per_order.head()

,order_id,n_items,product_value,freight_value,n_sellers
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,1


In [8]:
payments_per_order = payments.groupby("order_id").agg(
    payment_value     = ("payment_value", "sum"),
    max_installments  = ("payment_installments", "max"),
    main_payment_type = ("payment_type", lambda s: s.mode().iloc[0]),
).reset_index()

payments_per_order.head()

,order_id,payment_value,max_installments,main_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,credit_card


In [9]:
print("Orders with more than one review:", (reviews["order_id"].value_counts() > 1).sum())

reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"])
reviews_per_order = (
    reviews.sort_values("review_answer_timestamp")
           .drop_duplicates("order_id", keep="last")      # keep the most recent review
           [["order_id", "review_score"]]
)
print("Reviews after keeping one per order:", len(reviews_per_order))

Orders with more than one review: 547
Reviews after keeping one per order: 98673


In [10]:
master = (
    delivered
    .merge(customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(items_per_order,    on="order_id", how="left")
    .merge(payments_per_order, on="order_id", how="left")
    .merge(reviews_per_order,  on="order_id", how="left")
)

print("Shape:", master.shape)
print("Duplicate order_ids:", master["order_id"].duplicated().sum())   # must be 0
master.isna().sum()

Shape: (96470, 22)
Duplicate order_ids: 0


order_id                           0
customer_id                        0
order_status                       0
order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       1
order_delivered_customer_date      0
order_estimated_delivery_date      0
delivery_days                      0
delay_days                         0
is_late                            0
customer_unique_id                 0
customer_city                      0
customer_state                     0
n_items                            0
product_value                      0
freight_value                      0
n_sellers                          0
payment_value                      1
max_installments                   1
main_payment_type                  1
review_score                     646
dtype: int64

In [11]:
master.to_csv(PROCESSED / "orders_master.csv", index=False)
print("Saved:", PROCESSED / "orders_master.csv")

Saved: ..\data\processed\orders_master.csv
